In [1]:
def mv_weights(Theta_hat, mu, target_return=0.01):
    """
    Compute Mean-Variance portfolio weights with target return.
    
    Solves the constrained optimization:
    min w' Sigma w  subject to  w' mu = target_return  and  w' 1 = 1
    
    Solution uses Lagrange multipliers with two constraints.
    
    Parameters:
    -----------
    Theta_hat : np.ndarray, shape (p, p)
        Precision matrix (Sigma^{-1})
    mu : np.ndarray, shape (p,)
        Expected returns
    target_return : float
        Target portfolio return (default: 0.01 = 1% monthly)
    long_only : bool
        If True, falls back to GMV if MV produces negative weights
    
    Returns:
    --------
    w_star : np.ndarray, shape (p,)
        Portfolio weights
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    # Compute key quantities
    A = ones_p @ Theta_hat @ ones_p  # 1' Theta 1
    B = ones_p @ Theta_hat @ mu       # 1' Theta mu  
    C = mu @ Theta_hat @ mu           # mu' Theta mu
    D = A * C - B * B                  # Determinant
    
    # Check for singularity
    if np.abs(D) < 1e-10:
        print('SINGULARITY')
        # System is singular, use GMV instead
        if np.abs(A) > 1e-10:
            w_star = (Theta_hat @ ones_p) / A
            return w_star
        else:
            return ones_p / p
    
    
    # Compute Lagrange multipliers
    lambda1 = (C - B * target_return) / D
    lambda2 = (A * target_return - B) / D
    
    # Compute weights: w = lambda1 * Theta^{-1} 1 + lambda2 * Theta^{-1} mu
    w_star = lambda1 * (Theta_hat @ ones_p) + lambda2 * (Theta_hat @ mu)
    
    return w_star

def msr_weights(Theta_hat, mu):
    """
    Compute Maximum Sharpe Ratio portfolio weights.
    
    The maximum Sharpe ratio portfolio solves:
    max (w' mu) / sqrt(w' Sigma w)
    
    Solution (when mu represents excess returns):
    w ∝ Sigma^{-1} mu = Theta mu
    
    Then normalize so that sum(w) = 1.
    
    Parameters:
    -----------
    Theta_hat : np.ndarray, shape (p, p)
        Precision matrix (Sigma^{-1})
    mu : np.ndarray, shape (p,)
        Expected excess returns
    
    Returns:
    --------
    w_star : np.ndarray, shape (p,)
        Portfolio weights (sum to 1)
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    # Compute unnormalized weights: w ∝ Theta mu
    w_unnorm = Theta_hat @ mu
    
    # Normalize to sum to 1
    weight_sum = np.sum(w_unnorm)
    
    if np.abs(weight_sum) < 1e-10:
        print('WARNING: Weight sum near zero, returning equal weights')
        return ones_p / p
    
    w_star = w_unnorm / weight_sum
    
    return w_star

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

def train_random_forest(df, train_start, train_end, features=['mom12m', 'mve', 'bm']):
    """Train Random Forest classifier on historical data to predict positive returns."""
    train_df = df[(df['datadate'] >= train_start) & (df['datadate'] <= train_end)].copy()
    
    train_df['target'] = (train_df['ret_fwd_1'] > 0).astype(int)
    train_df = train_df.dropna(subset=features + ['target'])
    
    X_train = train_df[features]
    y_train = train_df['target']
    
    # scaler = StandardScaler()
    # X_train_scaled = scaler.fit_transform(X_train)
    
    rf_model = RandomForestClassifier(
        n_estimators=100, 
        max_depth=5, 
        min_samples_leaf=50,
        random_state=42, 
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    
    print(f"  Training samples: {len(train_df)}")
    print(f"  Training accuracy: {rf_model.score(X_train, y_train):.4f}")
    
    return rf_model

def select_stocks_rf_quantiles(df, predict_date, rf_model, 
                               features=['mom12m', 'mve', 'bm'],
                               top_q=0.90, bottom_q=0.10):
    """Select top and bottom percentiles of stocks based on RF probability."""
    predict_df = df[df['datadate'] == predict_date].copy()
    predict_df = predict_df.dropna(subset=features)
    
    if len(predict_df) == 0:
        print(f"  ⚠ No stocks with complete data on {predict_date}")
        return []
    
    X_predict = predict_df[features]
    
    predict_df['prob_positive'] = rf_model.predict_proba(X_predict)[:, 1]
    
    upper_bound = predict_df['prob_positive'].quantile(top_q)
    lower_bound = predict_df['prob_positive'].quantile(bottom_q)
    
    selected_df = predict_df[
        (predict_df['prob_positive'] >= upper_bound) | 
        (predict_df['prob_positive'] <= lower_bound)
    ]
    
    selected_permnos = selected_df['permno'].tolist()
    
    print(f"  Stocks evaluated: {len(predict_df)}")
    print(f"  Upper bound (Top {100-top_q*100:.0f}%): {upper_bound:.4f}")
    print(f"  Lower bound (Bottom {bottom_q*100:.0f}%): {lower_bound:.4f}")
    print(f"  Stocks selected: {len(selected_permnos)}")
    
    return selected_permnos

In [3]:
def naive_nodewise_regression(Y_star, lambda_grid=None):
    """
    Implements Naive Nodewise Regression with GIC.
    """
    n, p = Y_star.shape
    Theta_hat = np.zeros((p, p))
    tau_squared = np.zeros(p)
    
    if lambda_grid is None:
        lambda_grid = np.logspace(-3, 1, 50)
    
    for j in range(p):
        y_j = Y_star[:, j]
        Y_minus_j = np.delete(Y_star, j, axis=1)
        
        best_gic = np.inf
        best_lambda = lambda_grid[0]
        best_gamma = None
        best_ssr = None
        
        for lam in lambda_grid:
            lasso = Lasso(alpha=2*lam, fit_intercept=False, max_iter=10000)
            lasso.fit(Y_minus_j, y_j)
            gamma_j = lasso.coef_
            
            residuals = y_j - Y_minus_j @ gamma_j
            ssr = np.sum(residuals ** 2)
            sigma_sq_lambda = ssr / n
            q_lambda = np.sum(np.abs(gamma_j) > 1e-8)
            
            if sigma_sq_lambda > 1e-10:
                gic = np.log(sigma_sq_lambda) + q_lambda * (np.log(p) / n) * np.log(np.log(n))
            else:
                gic = np.inf
            
            if gic < best_gic:
                best_gic = gic
                best_lambda = lam
                best_gamma = gamma_j.copy()
                best_ssr = ssr
        
        gamma_j_star = best_gamma
        tau_squared[j] = best_ssr / n + best_lambda * np.sum(np.abs(gamma_j_star))
        
        Theta_hat[j, j] = 1 / tau_squared[j]
        off_diag = -gamma_j_star / tau_squared[j]
        Theta_hat[j, :j] = off_diag[:j]
        Theta_hat[j, j+1:] = off_diag[j:]
    
    Theta_hat_sym = (Theta_hat + Theta_hat.T) / 2
    return Theta_hat_sym


def gmv_weights(Theta_hat):
    """
    Compute Global Minimum Variance portfolio weights.
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    numerator = Theta_hat @ ones_p
    denominator = ones_p @ Theta_hat @ ones_p
    
    if np.abs(denominator) < 1e-10:
        return ones_p / p
    
    w_star = numerator / denominator
    return w_star

In [4]:
def integrated_backtest(df,
                       test_start_date='2020-01-31',
                       test_end_date='2024-04-30',
                       rf_train_years=15,
                       features=['mom12m', 'mve', 'bm'],
                       top_q=0.90,
                       bottom_q=0.10,
                       lookback_window=180,
                       transaction_cost=0.005,
                       portfolio_type='all',  
                       mv_target_return=0.01,
                       verbose=True):
    """
    Integrated backtest combining Random Forest quantile selection 
    with nodewise regression portfolio optimization.
    """
    df = df.copy()
    df['datadate'] = pd.to_datetime(df['datadate'])
    all_dates = sorted(df['datadate'].unique())
    
    test_start_dt = pd.to_datetime(test_start_date)
    test_end_dt = pd.to_datetime(test_end_date)
    
    start_year = test_start_dt.year
    end_year = test_end_dt.year
    test_years = list(range(start_year, end_year + 1))
    
    portfolio_types = ['gmv', 'mv', 'msr'] if portfolio_type == 'all' else [portfolio_type]
    
    results_storage = {ptype: {
        'returns': [],
        'dates': [],
        'weights_list': [],
        'turnover_list': [],
        'gross_returns': [],
        'prev_weights_dict': {},
        'prev_oos_returns_dict': {},
        'prev_gross_return': 0.0
    } for ptype in portfolio_types}
    
    current_permnos = []
    
    if verbose:
        print("="*70)
        print("INTEGRATED BACKTEST: RANDOM FOREST + NODEWISE")
        print(f"Test Period: {test_start_date} to {test_end_date}")
        print("="*70)
    
    for year in test_years:
        retrain_date = pd.to_datetime(f'{year}-01-31')
        
        if retrain_date not in all_dates:
            print(f"\n⚠ Warning: {retrain_date} not in dataset, skipping year {year}")
            continue
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"YEAR {year}: RETRAINING RANDOM FOREST")
            print(f"{'='*70}")
        
        train_end = pd.to_datetime(f'{year-1}-12-31')
        train_start = pd.to_datetime(f'{year - rf_train_years}-01-31')
        
        if verbose:
            print(f"RF training period: {train_start.strftime('%Y-%m-%d')} to {train_end.strftime('%Y-%m-%d')}")
        
        # 1. Train Random Forest
        rf_model = train_random_forest(
            df, train_start, train_end, features=features
        )
        
        # 2. Select stocks using probability quantiles
        current_permnos = select_stocks_rf_quantiles(
            df, retrain_date, rf_model,
            features=features, top_q=top_q, bottom_q=bottom_q
        )
        
        if len(current_permnos) == 0:
            print(f"  ⚠ No stocks selected for {year}, skipping")
            continue
        
        year_start_date = retrain_date
        
        if year == end_year:
            year_end_date = test_end_dt
        else:
            year_end_date = pd.to_datetime(f'{year}-12-31')
            if year_end_date not in all_dates:
                year_dates = [d for d in all_dates if d.year == year]
                year_end_date = max(year_dates) if year_dates else year_start_date
        
        if year == start_year and test_start_dt > year_start_date:
            year_start_date = test_start_dt
        
        try:
            year_start_idx = all_dates.index(year_start_date)
            year_end_idx = all_dates.index(year_end_date)
        except ValueError as e:
            print(f"  ⚠ Date error: {e}")
            continue
        
        if verbose:
            print(f"\nRunning monthly rebalancing from {year_start_date.strftime('%Y-%m-%d')} "
                  f"to {year_end_date.strftime('%Y-%m-%d')}")
            print(f"{'='*70}")
        
        for t in range(year_start_idx, year_end_idx + 1):
            current_date = all_dates[t]
            
            if t < lookback_window:
                if verbose:
                    print(f"\n[{current_date.strftime('%Y-%m-%d')}] Skipping: insufficient lookback")
                continue
            
            window_start_date = all_dates[t - lookback_window]
            window_end_date = all_dates[t - 1]
            
            train_data = df[
                (df['datadate'] >= window_start_date) & 
                (df['datadate'] <= window_end_date) &
                (df['permno'].isin(current_permnos))
            ]
            
            returns_pivot = train_data.pivot(
                index='datadate', columns='permno', values='ret_fwd_1'
            )
            
            window_dates = all_dates[t - lookback_window : t]
            returns_pivot = returns_pivot.reindex(index=window_dates)
            
            nan_assets = returns_pivot.columns[returns_pivot.isna().any()]
            filtered_pivot = returns_pivot.drop(columns=nan_assets)
            
            current_assets = filtered_pivot.columns.tolist()
            Y = filtered_pivot.values
            n_train, p_current = Y.shape
            
            if verbose:
                month_num = t - year_start_idx + 1
                print(f"\n[Month {month_num}] {current_date.strftime('%Y-%m-%d')}")
                print(f"  Assets: {p_current}/{len(current_permnos)} with complete data")
            
            if n_train < lookback_window or p_current < 2:
                if verbose:
                    print(f"  ⚠ Insufficient data, using previous weights")
                new_weights_dict = {ptype: results_storage[ptype]['prev_weights_dict'].copy() 
                                   for ptype in portfolio_types}
            else:
                try:
                    Y_bar = Y.mean(axis=0)
                    Y_star = Y - Y_bar
                    
                    Theta_hat = naive_nodewise_regression(Y_star)
                    new_weights_dict = {}
                    
                    if 'gmv' in portfolio_types:
                        w_gmv = gmv_weights(Theta_hat)
                        new_weights_dict['gmv'] = {asset: w_gmv[i] for i, asset in enumerate(current_assets)}
                    
                    if 'mv' in portfolio_types or 'msr' in portfolio_types:
                        mu = Y_bar
                        if 'mv' in portfolio_types:
                            w_mv = mv_weights(Theta_hat, mu, target_return=mv_target_return)
                            new_weights_dict['mv'] = {asset: w_mv[i] for i, asset in enumerate(current_assets)}
                        
                        if 'msr' in portfolio_types:
                            w_msr = msr_weights(Theta_hat, mu)
                            new_weights_dict['msr'] = {asset: w_msr[i] for i, asset in enumerate(current_assets)}
                    
                    if verbose:
                        print(f"  ✓ Nodewise completed for {', '.join(portfolio_types)}")
                    
                except Exception as e:
                    if verbose:
                        print(f"  ✗ Error: {e}")
                    new_weights_dict = {ptype: results_storage[ptype]['prev_weights_dict'].copy() 
                                       for ptype in portfolio_types}
            
            for ptype in portfolio_types:
                weights = new_weights_dict[ptype]
                weight_sum = sum(weights.values())
                if weight_sum > 1e-10:
                    weights = {k: v/weight_sum for k, v in weights.items()}
                else:
                    weights = results_storage[ptype]['prev_weights_dict'].copy()
            
            oos_data = df[df['datadate'] == current_date]
            oos_returns_series = oos_data.set_index('permno')['ret_fwd_1'].dropna()
            oos_returns_dict = oos_returns_series.to_dict()
            
            for ptype in portfolio_types:
                weights = new_weights_dict[ptype]
                prev_weights = results_storage[ptype]['prev_weights_dict']
                prev_oos_returns = results_storage[ptype]['prev_oos_returns_dict']
                prev_gross_ret = results_storage[ptype]['prev_gross_return']
                
                common_assets = set(weights.keys()) & set(oos_returns_dict.keys())
                
                if len(common_assets) == 0:
                    continue
                
                common_weights = {a: weights[a] for a in common_assets}
                common_weight_sum = sum(common_weights.values())
                if common_weight_sum > 1e-10:
                    common_weights = {k: v/common_weight_sum for k, v in common_weights.items()}
                else:
                    continue
                
                gross_return = sum(common_weights[a] * oos_returns_dict[a] for a in common_assets)
                
                if np.isnan(gross_return) or np.isinf(gross_return):
                    continue
                
                if len(prev_weights) > 0:
                    adjusted_prev = {}
                    for asset, prev_w in prev_weights.items():
                        if asset in prev_oos_returns:
                            prev_r = prev_oos_returns[asset]
                            if abs(1 + prev_gross_ret) > 1e-6:
                                adjusted_prev[asset] = prev_w * (1 + prev_r) / (1 + prev_gross_ret)
                            else:
                                adjusted_prev[asset] = 0.0
                        else:
                            if abs(1 + prev_gross_ret) > 1e-6:
                                adjusted_prev[asset] = prev_w / (1 + prev_gross_ret)
                            else:
                                adjusted_prev[asset] = 0.0
                    
                    all_assets = set(adjusted_prev.keys()) | set(common_weights.keys())
                    turnover = sum(abs(common_weights.get(a, 0.0) - adjusted_prev.get(a, 0.0)) for a in all_assets)
                    tc = transaction_cost * (1 + gross_return) * turnover
                else:
                    turnover = sum(abs(w) for w in common_weights.values())
                    tc = transaction_cost * (1 + gross_return) * turnover
                
                net_return = gross_return - tc
                
                results_storage[ptype]['returns'].append(net_return)
                results_storage[ptype]['dates'].append(current_date)
                results_storage[ptype]['weights_list'].append(common_weights.copy())
                results_storage[ptype]['turnover_list'].append(turnover)
                results_storage[ptype]['gross_returns'].append(gross_return)
                
                results_storage[ptype]['prev_weights_dict'] = common_weights.copy()
                results_storage[ptype]['prev_oos_returns_dict'] = {a: oos_returns_dict[a] for a in common_assets}
                results_storage[ptype]['prev_gross_return'] = gross_return
            
            if verbose:
                print(f"  Portfolio Returns:")
                for ptype in portfolio_types:
                    if len(results_storage[ptype]['returns']) > 0:
                        last_idx = len(results_storage[ptype]['returns']) - 1
                        gross_ret = results_storage[ptype]['gross_returns'][last_idx]
                        net_ret = results_storage[ptype]['returns'][last_idx]
                        to = results_storage[ptype]['turnover_list'][last_idx]
                        tc = gross_ret - net_ret
                        print(f"    {ptype.upper()}: Gross={gross_ret:>7.4f} | TO={to:>5.3f} | TC={tc:>7.5f} | Net={net_ret:>7.4f}")
    
    if verbose:
        print("\n" + "="*70)
        print("BACKTEST COMPLETE")
        print("="*70)
    
    results_dict = {}
    
    for ptype in portfolio_types:
        portfolio_returns = results_storage[ptype]['returns']
        portfolio_dates = results_storage[ptype]['dates']
        portfolio_turnover_list = results_storage[ptype]['turnover_list']
        portfolio_gross_returns = results_storage[ptype]['gross_returns']
        
        if len(portfolio_returns) == 0:
            results_dict[ptype] = {'results_df': pd.DataFrame(), 'metrics': {}}
            continue
        
        results_df = pd.DataFrame({
            'date': portfolio_dates,
            'portfolio_return': portfolio_returns,
            'portfolio_gross_return': portfolio_gross_returns,
            'portfolio_turnover': portfolio_turnover_list
        })
        results_df['cumulative_return'] = (1 + results_df['portfolio_return']).cumprod() - 1
        
        mean_return = np.mean(portfolio_returns)
        variance = np.var(portfolio_returns, ddof=1)
        sharpe_ratio = mean_return / np.sqrt(variance) if variance > 0 else 0
        
        annual_return = mean_return * 12
        annual_volatility = np.sqrt(variance * 12)
        annual_sharpe = annual_return / annual_volatility if annual_volatility > 0 else 0
        
        overall_metrics = {
            'mean_return': mean_return,
            'variance': variance,
            'sharpe_ratio': sharpe_ratio,
            'annual_return': annual_return,
            'annual_volatility': annual_volatility,
            'annual_sharpe_ratio': annual_sharpe,
            'total_return': results_df['cumulative_return'].iloc[-1],
            'avg_turnover': np.mean(portfolio_turnover_list),
            'n_periods': len(portfolio_returns)
        }
        
        results_dict[ptype] = {
            'results_df': results_df,
            'metrics': overall_metrics
        }
    
    return results_dict

In [5]:
df = pd.read_csv('../green cleaned.csv', dtype={'ncusip': 'string'})
df['ret_fwd_1'] = (df.groupby('permno')['ret_excess'].shift(-1) )

In [6]:
variables = ['agr', 'bm', 'mom12m', 'mve', 'operprof', 'roeq', 'absacc',
            'acc', 'aeavol', 'age', 'baspread', 'BETA', 'bm_ia', 'cash',
            'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho',
            'chempia', 'chfeps', 'chinv', 'chmom', 'chnanalyst', 'chpmia',
            'chtx', 'cinvest', 'convind', 'currat', 'depr', 'disp', 'divi',
            'divo', 'dy', 'ear', 'egr', 'ep', 'fgr5yr', 'gma', 'grcapx',
            'grltnoa', 'herf', 'hire', 'idiovol', 'ill', 'indmom', 'invest',
            'IPO', 'lev', 'mom1m', 'mom36m', 'ms', 'mve_ia', 'nanalyst',
            'nincr', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr',
            'pchgm_pchsale', 'pchsale_pchinvt', 'pchsale_pchrect',
            'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'pricedelay', 'ps',
            'rd', 'rd_mve', 'rd_sale', 'realestate', 'retvol', 'roaq',
            'roavol', 'roic', 'rsup', 'salecash', 'saleinv', 'salerec',
            'secured', 'securedind', 'sfe', 'sgr', 'sin', 'sp', 'std_dolvol',
            'std_turn', 'stdcf', 'sue', 'tang', 'tb', 'turn', 'zerotrade']

In [7]:
results = integrated_backtest(
    df,
    test_start_date='2021-10-31',
    test_end_date='2024-04-30',
    rf_train_years=15,
    features=variables,#['mom12m', 'mve', 'bm'],
    top_q=0.90,
    bottom_q=0.10,
    lookback_window=180,
    transaction_cost=0.001,
    verbose=True
)

# Access results for each portfolio
gmv_results = results['gmv']['results_df']
gmv_metrics = results['gmv']['metrics']

mv_results = results['mv']['results_df']
mv_metrics = results['mv']['metrics']

msr_results = results['msr']['results_df']
msr_metrics = results['msr']['metrics']

# Compare Sharpe ratios
print(f"GMV Sharpe: {gmv_metrics['annual_sharpe_ratio']:.4f}")
print(f"MV Sharpe:  {mv_metrics['annual_sharpe_ratio']:.4f}")
print(f"MSR Sharpe: {msr_metrics['annual_sharpe_ratio']:.4f}")

INTEGRATED BACKTEST: RANDOM FOREST + NODEWISE
Test Period: 2021-10-31 to 2024-04-30

YEAR 2021: RETRAINING RANDOM FOREST
RF training period: 2006-01-31 to 2020-12-31
  Training samples: 89754
  Training accuracy: 0.5856
  Stocks evaluated: 497
  Upper bound (Top 10%): 0.5714
  Lower bound (Bottom 10%): 0.5404
  Stocks selected: 100

Running monthly rebalancing from 2021-10-31 to 2021-12-31

[Month 1] 2021-10-31
  Assets: 51/100 with complete data
  ✓ Nodewise completed for gmv, mv, msr
  Portfolio Returns:
    GMV: Gross=-0.0181 | TO=1.092 | TC=0.00107 | Net=-0.0191
    MV: Gross=-0.0192 | TO=1.094 | TC=0.00107 | Net=-0.0203
    MSR: Gross=-0.0126 | TO=1.114 | TC=0.00110 | Net=-0.0137

[Month 2] 2021-11-30
  Assets: 52/100 with complete data
  ✓ Nodewise completed for gmv, mv, msr
  Portfolio Returns:
    GMV: Gross= 0.0784 | TO=0.072 | TC=0.00008 | Net= 0.0783
    MV: Gross= 0.0786 | TO=0.089 | TC=0.00010 | Net= 0.0785
    MSR: Gross= 0.0758 | TO=0.069 | TC=0.00007 | Net= 0.0757

[Mon

In [9]:
print(f"\n GMV")
print(f"Annualized Sharpe Ratio: {gmv_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {gmv_metrics['mean_return']*12:.4f}")
print(f"Variance: {gmv_metrics['variance']*12:.4f}")
print(f"Avg Turnover: {gmv_metrics['avg_turnover']:.4f}")

print(f"\n MV")
print(f"Annualized Sharpe Ratio: {mv_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {mv_metrics['mean_return']*12:.4f}")
print(f"Variance: {mv_metrics['variance']*12:.4f}")
print(f"Avg Turnover: {mv_metrics['avg_turnover']:.4f}")

print(f"\n MSR")
print(f"Annualized Sharpe Ratio: {msr_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {msr_metrics['mean_return']*12:.4f}")
print(f"Variance: {msr_metrics['variance']*12:.4f}")
print(f"Avg Turnover: {msr_metrics['avg_turnover']:.4f}")


 GMV
Annualized Sharpe Ratio: 0.1356
Mean Return: 0.0244
Variance: 0.0324
Avg Turnover: 0.2710

 MV
Annualized Sharpe Ratio: 0.0764
Mean Return: 0.0138
Variance: 0.0327
Avg Turnover: 0.3096

 MSR
Annualized Sharpe Ratio: 0.2468
Mean Return: 0.0435
Variance: 0.0311
Avg Turnover: 0.2770
